#### Algorithm description:
This algorithm is takes the peak array as input. The peak array is a $f\times t$ matrix representing a spectrum where f=frequency bins and t=time steps. The matrix consists of 1s where the corresponding frequency and time step has high power correspnding to a center tone peak and 0s at other indexes. 

This matric is modified to construct "curr_peaks", which consists of the $(x,y)$ coordinate of each peak corresponding to (time step,frequency bin). This is the array that the hungarian algorithm operates on.

![Visual representation of curr_peak](peak_arr_img.png)

Initialization: When the algorithm begins, a new DopplerTrack (object of class DopplerTrack) is created for each point at the starting time step. Each DopplerTrack object is equivalent to a new doppler curve adn is identified by its track_id. Each DopplerTrack object also has the following parameters: track_points (associated peak points), staleness and staleness_limit. The purpose of these parameters will (hopefully) be more clear after reading this explanation

Update: At the consecutive time steps/iterations, new points/peaks are assigned to existing tracks based on eucledian distance. `assign_peaks_to_tracks(peaks, tracks, dist_thresh)` is the function that handles this assignment of peaks to existing tracks based on linear sum assignment. The cost matrix in this function is the matrix of distances between all latest points in the existing tracks and all new points that need to be assigned to the existing tracks.

The parameter dist_thresh is a threshold used to decide if a candidate matching can be finalized. If a point $P(x,y)$ is assigned to an existing track T based on linear sum assignment, but the point is $>dist\_thresh$ away from the last point in track T, then the point is not assigned to an existing track. It remains unmatched.

All unmatched points are assigned to new tracks. 

All tracks that are not updated in the current iteration are called unmatched_tracks. For each consective time step/iteration that the track is unmatched (meaning no new point/peak is assigned to it), its staleness is incremented. max_time_till_no_update is the threshold for staleness. It is the max number of iterations till it is ok for a track to remain unmatched. After this, the track is declared dead and can no longer be updated. In our case, this indicates that the satellite may have passed because the center tone peak associated with it is no longer visible so we no longer update the set of points associated with the satellite's doppler track.

In [1]:
from math import inf
from telnetlib import DO
from scipy import io as sio
from scipy.optimize import linear_sum_assignment
import numpy as np

peak_file_path = 'peak_array_9_12_11_27_5.mat'
peak_array = sio.loadmat(peak_file_path)['peak_array']


def assign_peaks_to_tracks(peaks, tracks, dist_thresh):
    # fetch the last point from all track and create an array of "curr_points"
    curr_points = []
    for trk in tracks:
        curr_points.append(trk.track_points[-1])
    
    curr_points = np.array(curr_points) # latest point in the existinf active DopplerTrack objects
    cost_matrix = np.empty((len(curr_points), len(peaks))) # constructing cost matrix of distances between curr_points and new points
    for i,curr_point in enumerate(curr_points):
        for j,next_point in enumerate(peaks):
                cost_matrix[i,j] = np.linalg.norm(curr_point-next_point)
    
    if min(cost_matrix.shape) > 0:
        a = (cost_matrix < dist_thresh).astype(np.int32)
        if a.sum(1).max() == 1 and a.sum(0).max() == 1:
            matched_indices = np.stack(np.where(a), axis=1)
        else:
            x, y = linear_sum_assignment(cost_matrix)
            matched_indices = np.array(list(zip(x, y)))
    else:
        matched_indices = np.empty(shape=(0,2))
    
    # getting points that are not matched to any DopplerTrack
    unmatched_peaks = [] 
    for p, peak in enumerate(peaks):
        if(p not in matched_indices[:,1]):
            unmatched_peaks.append(p)
    unmatched_tracks = []
    for t, trk in enumerate(tracks):
        if(t not in matched_indices[:,0]):
            unmatched_tracks.append(t)

    #filter out matched points if the distance is >dist_thresh
    matches = []
    for m in matched_indices:
        if(cost_matrix[m[0], m[1]]>dist_thresh):
            unmatched_tracks.append(m[0])
            unmatched_peaks.append(m[1])
        else:
            matches.append(m.reshape(1,2))

    if(len(matches)==0):
        matches = np.empty((0,2),dtype=int)
    else:
        matches = np.concatenate(matches,axis=0)
    
    return matches, np.array(unmatched_peaks), np.array(unmatched_tracks)
    

class DopplerTrack(object):
    num_tracks = 0
    def __init__(self, stalesness_limit=25) -> None:
        self.track_id = DopplerTrack.num_tracks
        DopplerTrack.num_tracks += 1
        self.track_points = []
        self.staleness = 0
        self.staleness_limit = stalesness_limit

    def update(self, peak) -> None:
        # add peak to the track_points of DopplerTrack if there are elements in peak
        if len(peak) != 0 :
            self.track_points.append(peak)
            self.staleness = 0
        # if not, then increment the staleness
        # check if staleness limit is exceeded, if it has, then return "remove"
        else:
            self.staleness+=1

## Algorithm description: This algorithm takes the peak array as input. The peak array is a 2D array 

# TODO: set these parameters:
time = 1 # set start time (not used currently)
step = 0.5 # time step increment at each iteration
max_dist = 30 #
curr_tracks = []
all_tracks = [] 
fbin_hardware = 517
max_time_till_no_update = 30

for i in range(np.shape(peak_array)[0]):
    curr_peaks = np.vstack((np.ones(len(peak_array[i,:]))*time+step*(i-1), peak_array[i,:])).T
    curr_peaks =  np.array([curr_peaks[np.where(curr_peaks[:, 1] != 0), 0][0], np.where(curr_peaks[:, 1] != 0)[0]]).T
    curr_peaks = curr_peaks[np.where(curr_peaks[:, 1] != fbin_hardware)]
    
    if len(curr_tracks) == 0:
        # create new tracks for each peak
        for peak in curr_peaks:
            trk = DopplerTrack(stalesness_limit=max_time_till_no_update)
            trk.track_points.append(peak.tolist())
            curr_tracks.append(trk)
            
    else:
        matched, unmatched_peaks, unmatched_trks = assign_peaks_to_tracks(curr_peaks, curr_tracks, max_dist)
        for elem in matched:
            curr_tracks[elem[0]].update(curr_peaks[elem[1]].tolist()) # see if matched is a list
        
        for points in unmatched_peaks:
            # create new tracks
            trk = DopplerTrack(stalesness_limit=max_time_till_no_update)
            curr_tracks.append(trk)
            trk.update(curr_peaks[points].tolist())
        
        for trks in unmatched_trks:
            # increament track staleness
            curr_tracks[trks].update([])

    # remove stale tracks
    for i,track in enumerate(curr_tracks):
        if track.staleness > track.staleness_limit:
            all_tracks.append(curr_tracks.pop(i))

for i,track in enumerate(curr_tracks):
    all_tracks.append(curr_tracks.pop(i))

track_dict = {}
for track in all_tracks:
    track_dict["track_"+str(track.track_id)] = track.track_points
    print(track.track_id, ":", track.track_points, "\n")



output_tracks_file = "./output_tracks_.mat"
sio.savemat(output_tracks_file, {'tracks_dict': track_dict})

FileNotFoundError: [Errno 2] No such file or directory: 'peak_array_9_12_11_27_5.mat'